# 201 · Schema evolution lab

Companion to [Schema evolution](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/201/schema-evolution/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/201/schema_evolution.ipynb)

**Goal:** practice additive evolution with a mini Protobuf-style codec—old readers skip unknown fields; new readers default missing fields.

**Why this lab:** rolling deploys mean old readers meet new writers (and the reverse). Evolution is a **policy + encoding rules**, not a checkbox.

**How to use:** define v1/v2 messages, run both compatibility directions, then glance at JSON additive behavior.

**Expect:** writer v2 → reader v1 keeps `id`/`name` and ignores `email`; writer v1 → reader v2 gets empty default `email`.

> **Honesty banner:** sizes and timings here are **illustrative**. Suite [Results](https://leo-gan.github.io/GLD.SerializerBenchmark/) own harness truth. Compare within one language and paradigm—not global format rankings.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Optional, Tuple


class WireError(Exception):
    pass


def encode_varint(u: int) -> bytes:
    out = bytearray()
    while u > 0x7F:
        out.append((u & 0x7F) | 0x80)
        u >>= 7
    out.append(u & 0x7F)
    return bytes(out)


def decode_varint(buf: bytes, i: int = 0) -> Tuple[int, int]:
    value = shift = nread = 0
    while True:
        if i >= len(buf):
            raise WireError("truncated")
        b = buf[i]
        i += 1
        nread += 1
        if nread > 10:
            raise WireError("overlong")
        value |= (b & 0x7F) << shift
        if b < 0x80:
            return value, i
        shift += 7


def encode_key(fn: int, wt: int) -> bytes:
    return encode_varint((fn << 3) | wt)


def require(buf: bytes, i: int, n: int):
    if i + n > len(buf):
        raise WireError("truncated payload")
    return buf[i : i + n], i + n



## v1 and v2 messages

**Why:** a minimal schema change is easier to reason about than a production `.proto` history.

**How:** v1 has `id=1`, `name=2`; v2 **adds** optional `email=3` without reusing numbers.

**Expect:** encoders omit empty defaults (proto3-style); decoders accept fields in any order.

**Why it matters:** never repurpose field numbers—semantic overwrite is a silent production break.


In [ ]:
@dataclass
class UserV1:
    id: int = 0
    name: str = ""


@dataclass
class UserV2:
    id: int = 0
    name: str = ""
    email: str = ""  # new optional field


def encode_v1(u: UserV1) -> bytes:
    out = bytearray()
    if u.id:
        out += encode_key(1, 0) + encode_varint(u.id)
    if u.name:
        b = u.name.encode()
        out += encode_key(2, 2) + encode_varint(len(b)) + b
    return bytes(out)


def encode_v2(u: UserV2) -> bytes:
    out = bytearray(encode_v1(UserV1(u.id, u.name)))
    if u.email:
        b = u.email.encode()
        out += encode_key(3, 2) + encode_varint(len(b)) + b
    return bytes(out)


def decode_v1(buf: bytes) -> UserV1:
    i = 0
    u = UserV1()
    while i < len(buf):
        key, i = decode_varint(buf, i)
        fn, wt = key >> 3, key & 7
        if wt == 0:
            v, i = decode_varint(buf, i)
            if fn == 1:
                u.id = v
        elif wt == 2:
            n, i = decode_varint(buf, i)
            payload, i = require(buf, i, n)
            if fn == 2:
                u.name = payload.decode()
            # fn == 3 or other: skip (old reader + new writer)
        elif wt == 1:
            _, i = require(buf, i, 8)
        elif wt == 5:
            _, i = require(buf, i, 4)
        else:
            raise WireError("bad wt")
    return u


def decode_v2(buf: bytes) -> UserV2:
    i = 0
    u = UserV2()
    while i < len(buf):
        key, i = decode_varint(buf, i)
        fn, wt = key >> 3, key & 7
        if wt == 0:
            v, i = decode_varint(buf, i)
            if fn == 1:
                u.id = v
        elif wt == 2:
            n, i = decode_varint(buf, i)
            payload, i = require(buf, i, n)
            if fn == 2:
                u.name = payload.decode()
            elif fn == 3:
                u.email = payload.decode()
        elif wt == 1:
            _, i = require(buf, i, 8)
        elif wt == 5:
            _, i = require(buf, i, 4)
        else:
            raise WireError("bad wt")
    return u



## Scenarios

**Why:** both deploy directions happen in real fleets.

**How:** encode with v2, decode with v1; encode with v1, decode with v2.

**Expect:** old reader ignores unknown `email`; new reader defaults missing `email` to empty.

**Why it matters:** if your deploy cannot guarantee “all readers first,” you need both directions for a support window (full compatibility).


In [ ]:
# New writer → old reader: old reader must ignore email
v2_bytes = encode_v2(UserV2(id=1, name="Ada", email="ada@example.com"))
old_view = decode_v1(v2_bytes)
assert old_view == UserV1(id=1, name="Ada")
print("OK writer v2 → reader v1:", old_view)

# Old writer → new reader: email defaults empty
v1_bytes = encode_v1(UserV1(id=1, name="Ada"))
new_view = decode_v2(v1_bytes)
assert new_view == UserV2(id=1, name="Ada", email="")
print("OK writer v1 → reader v2:", new_view)

# Breaking pattern demo: reusing field 2 for email would corrupt name — DO NOT
print("Never repurpose field numbers or JSON property meanings in place.")



## JSON additive evolution (contrast)

**Why:** many JSON services evolve by “extra properties are fine.”

**How:** parse a v2-shaped object with a v1 reader that only picks known keys.

**Expect:** `email` ignored; `id`/`name` preserved—similar *additive* idea, weaker type/identity discipline than field numbers.

**Why it matters:** removing or renaming JSON properties without a plan still breaks consumers; additive-only is the safe default.


In [ ]:
import json

def read_v1_json(s: str) -> dict:
    d = json.loads(s)
    return {"id": d.get("id", 0), "name": d.get("name", "")}


payload_v2 = json.dumps({"id": 1, "name": "Ada", "email": "ada@example.com"})
print("v1 view of v2 JSON:", read_v1_json(payload_v2))



## Takeaways

- Prefer **additive** optional fields; document defaults.
- Never reuse Protobuf field numbers or silently change meaning.
- Forward/backward are **directions**—know your deploy order.

**Why it matters:** most “random” consumer breakage in polyglot systems is evolution policy failure, not a slow codec.

**Next:** [Dynamic vs IDL binary](./dynamic_vs_idl_binary.ipynb)
